In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

data_path = os.path.join(path, 'Q3_data.csv')
data = pd.read_csv(data_path)
print(f"Shape: {data.shape}")

In [ ]:
# Task 2: Write your code here:
data.head()

In [ ]:
# Task 3: Write your code here:
data.info()

In [ ]:
# Task 4: Write your code here:
data.info()

In [ ]:
# Task 1: Write your code here:
cols = data.columns
data_clean = data[cols].copy()
data_clean_cols = data_clean.columns
# data_clean = data_clean.dropna(subset=data_clean_cols)

In [ ]:
# Task 2: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = data_clean.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    data_clean.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

X = data_clean
y = data_clean['Target']


print("Applying Label Encoding to target variable 'y'...")
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Applying One-Hot Encoding to feature DataFrame 'X'...")
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_encoded = pd.DataFrame(onehot_encoder.fit_transform(X), columns=onehot_encoder.get_feature_names_out(X.columns))


print("Verification of encoded data shapes:")
print(f"Shape of X_encoded: {X_encoded.shape}")
print(f"Shape of y_encoded: {y_encoded.shape}")

print("First 5 rows of X_encoded:")
display(X_encoded.head())
print("First 5 elements of y_encoded:")
print(y_encoded[:5])

In [ ]:
# Task 4: Write your code here:
print("Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_encoded, test_size=0.3, random_state=42)

print("Data split successful.")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

In [ ]:
# Task 5: Write your code here:
import seaborn as sns

def check_target_imbalance(data_clean, target_column):
  print("Target Distribution:")
  print(data_clean[target_column].value_counts(normalize=True))
  sns.countplot(x=data_clean[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(data_clean, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
# ----------------

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

# ----------------

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score


model = CatBoostClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
print(f"Accuracy: {accuracy}")
print(f"F1: {f1}")


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': data_clean_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
fi = feature_importance['importance']
print(fi)

In [ ]:
# Task Bonus: Write your code here:
X = feature_importance['importance']


# ------------------

# Task Bonus: Write your code here:
from sklearn.model_selection import StratifiedKFold

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

models = {
  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )
}

results = {}
n_splits = 3
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for model_name in models:
  results[model_name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():

    print(f"Training {model_name}...")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    results[model_name]['f1'].append(f1)

